In [6]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [7]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 51.5,
	"longitude": 4.5,
	"start_date": "2026-03-01",
	"end_date": "2026-04-16",
	"daily": ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "daylight_duration", "sunshine_duration", "sunset", "sunrise", "weather_code", "shortwave_radiation_sum", "apparent_temperature_min", "apparent_temperature_mean", "apparent_temperature_max"],
	"hourly": ["temperature_2m", "soil_temperature_0_to_7cm", "weather_code", "shortwave_radiation_instant", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high"],
	"timezone": "Europe/Berlin",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 51.49384689331055°N 4.5652174949646°E
Elevation: 13.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s


In [8]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_soil_temperature_0_to_7cm = hourly.Variables(1).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(2).ValuesAsNumpy()
hourly_shortwave_radiation_instant = hourly.Variables(3).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(4).ValuesAsNumpy()
hourly_cloud_cover_low = hourly.Variables(5).ValuesAsNumpy()
hourly_cloud_cover_mid = hourly.Variables(6).ValuesAsNumpy()
hourly_cloud_cover_high = hourly.Variables(7).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["soil_temperature_0_to_7cm"] = hourly_soil_temperature_0_to_7cm
hourly_data["weather_code"] = hourly_weather_code
hourly_data["shortwave_radiation_instant"] = hourly_shortwave_radiation_instant
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["cloud_cover_low"] = hourly_cloud_cover_low
hourly_data["cloud_cover_mid"] = hourly_cloud_cover_mid
hourly_data["cloud_cover_high"] = hourly_cloud_cover_high

hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe

,date,temperature_2m,soil_temperature_0_to_7cm,weather_code,shortwave_radiation_instant,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high
0,2026-02-28 22:00:00+00:00,6.15,7.25,1.0,0.000000,23.0,9.0,23.0,0.0
1,2026-02-28 23:00:00+00:00,5.15,6.75,0.0,0.000000,1.0,1.0,0.0,0.0
2,2026-03-01 00:00:00+00:00,4.65,6.50,0.0,0.000000,0.0,0.0,0.0,0.0
3,2026-03-01 01:00:00+00:00,4.30,6.00,3.0,0.000000,97.0,0.0,0.0,97.0
4,2026-03-01 02:00:00+00:00,4.10,5.70,0.0,0.000000,5.0,0.0,4.0,2.0
...,...,...,...,...,...,...,...,...,...
1123,2026-04-16 17:00:00+00:00,14.90,15.65,2.0,126.088264,73.0,16.0,30.0,59.0
1124,2026-04-16 18:00:00+00:00,14.05,15.00,2.0,46.175938,66.0,28.0,17.0,49.0
1125,2026-04-16 19:00:00+00:00,12.30,13.55,1.0,0.000000,41.0,23.0,3.0,21.0
1126,2026-04-16 20:00:00+00:00,11.30,12.20,1.0,0.000000,20.0,6.0,5.0,10.0


In [ ]:
hourly_dataframe.to_csv("data/2026_march_april_hourly_historical.csv", index = False)